In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

try:
    from lightgbm import LGBMClassifier
    lightgbm_available = True
except ImportError:
    lightgbm_available = False

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import log_loss, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib


print("\n--- Step 2: Loading and Combining Data ---")

df_benign = pd.read_csv("benign_noip_flat.csv")
df_malicious = pd.read_csv("malicious_noip_flat.csv")

df_combined = pd.concat([df_benign, df_malicious], ignore_index=True)

print(f"  - Total Rows: {df_combined.shape[0]}")
print(f"  - Total Columns: {df_combined.shape[1]}")

print("\nFirst 5 rows of the combined data:")
print(df_combined.head())

print("\nDataset Info (Columns, Non-Null Counts, and Data Types):")
df_combined.info()

print("\n--- Step 3: Defining Features (X) and Target (y) ---")

y = df_combined['label'].map({'benign': 0, 'malicious': 1})

categorical_features = ['agent_name', 'srcuser', 'decoder_name', 'program_name', 'rule_groups', 'rule_description']
numerical_features = ['rule_level', 'hour_of_day', 'day_of_week']

X = df_combined[categorical_features + numerical_features]
print("Features (X) and Target (y) are defined.")
print(f"Using {len(categorical_features)} categorical and {len(numerical_features)} numerical features.")

print("\n--- Step 4: Splitting Data into Train and Test Sets ---")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

total_samples = len(X)
train_samples = len(X_train)
test_samples = len(X_test)

print(f"Total samples split: {total_samples}")
print(f"  - Training samples: {train_samples} (~{train_samples/total_samples*100:.0f}%)")
print(f"  - Testing samples:  {test_samples} (~{test_samples/total_samples*100:.0f}%)")

print("\n--- Step 5: Creating Preprocessing Pipeline ---")
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
])


print("\n--- Step 6: Defining, Training, and Evaluating Models ---")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

if lightgbm_available:
    models["LightGBM"] = LGBMClassifier(random_state=42, verbosity=-1)

results = {}

loss_scores = {}

for model_name, model in models.items():

    clf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('classifier', model)])

    print(f"\n--- Training {model_name} ---")

    clf_pipeline.fit(X_train, y_train)

    print(f"--- Evaluating {model_name} ---")

    y_train_pred = clf_pipeline.predict(X_train)
    y_train_proba = clf_pipeline.predict_proba(X_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    train_loss = log_loss(y_train, y_train_proba)

    y_test_pred = clf_pipeline.predict(X_test)
    y_test_proba = clf_pipeline.predict_proba(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_loss = log_loss(y_test, y_test_proba)

    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    print(f"Performance Metrics (Checking for Overfitting):")
    print(f"  - Training Accuracy: {train_accuracy:.4f}  |  Testing Accuracy: {test_accuracy:.4f}")
    print(f"  - Training Loss:     {train_loss:.4f}  |  Testing Loss:     {test_loss:.4f}")
    print("\n" + "-"*20)

    results[model_name] = {
        'Accuracy': test_accuracy,
        'Precision': test_precision,
        'Recall': test_recall,
        'F1-Score': test_f1
    }

    loss_scores[model_name] = {'train': train_loss, 'test': test_loss}

    print("Full Test Set Evaluation:")
    report = classification_report(y_test, y_test_pred, target_names=['benign (0)', 'malicious (1)'])
    cm = confusion_matrix(y_test, y_test_pred)

    print("Classification Report:")
    print(report)
    print("Confusion Matrix:")
    print(cm)

    print("\n--- Example of Final Output Format (Head of Test Set) ---")

    malicious_scores = y_test_proba[:, 1]

    label_map = {0: 'benign', 1: 'malicious'}
    predicted_labels = pd.Series(y_test_pred).map(label_map)

    output_df = pd.DataFrame({
        'label': predicted_labels,
        'score (prob_malicious)': malicious_scores
    })

    print(output_df.head().to_string())

    print("\n" + "="*30)

print("\n--- Step 7: Interpretation of Overfitting/Underfitting ---")

print("--- Model Loss Comparison ---")
for model_name, losses in loss_scores.items():
    print(f"  {model_name}:")
    print(f"    - Train/Test Loss Delta: {abs(losses['train'] - losses['test']):.4f}")


print("\n--- Step 8: Final Model Score Summary (Test Set) ---")
results_df = pd.DataFrame(results).T
print(results_df.to_markdown(floatfmt=".4f"))

print("\n--- Step 9: Data Visualization and Model  ---")

sns.set_theme(style="whitegrid", palette="muted")

try:
    ax1 = results_df.plot(kind='bar', figsize=(12, 7), title="Model Performance Comparison (Test Set)")
    ax1.set_ylabel("Score (Higher is Better)")
    ax1.set_xlabel("Model")
    ax1.tick_params(axis='x', rotation=0)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig('model_scores_comparison.png')
    plt.close()
    print("\nSuccessfully saved 'model_scores_comparison.png'")

    loss_df = pd.DataFrame(loss_scores).T
    loss_df.columns = ['Training Loss', 'Test Loss']

    ax2 = loss_df.plot(kind='bar', figsize=(10, 6), title="Model Loss (Training vs. Test)")
    ax2.set_ylabel("Log Loss (Lower is Better)")
    ax2.set_xlabel("Model")
    ax2.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.savefig('model_loss_comparison.png')
    plt.close()
    print("Successfully saved 'model_loss_comparison.png'")

    best_model_name = results_df['Accuracy'].idxmax()
    best_accuracy = results_df['Accuracy'].max()

    print("\n--- Best Model Recommendation ---")
    print(f"Based on the 'Accuracy' on the test set:")
    print(f"  - The best performing model is: {best_model_name} (Test Accuracy: {best_accuracy:.4f})")
    print("\n  - Review 'model_scores_comparison.png' for a full comparison of all metrics.")
    print("  - Review 'model_loss_comparison.png' to see the train vs. test loss for all models.")

except Exception as e:
    print(f"\nAn error occurred during plotting: {e}")

print("\n--- Step 10: Saving the Best Model for API Use ---")

try:
    best_model_name = results_df['Accuracy'].idxmax()

    best_model_object = models[best_model_name]

    print(f"The best model was: {best_model_name}. Retraining it on ALL data for production...")

    final_production_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', best_model_object)
    ])

    final_production_pipeline.fit(X, y)

    model_filename = 'malware_model.joblib'
    joblib.dump(final_production_pipeline, model_filename)

    print(f"\nSuccessfully trained and saved final model to: '{model_filename}'")
    print("This file can now be loaded by your API for predictions.")

except Exception as e:
    print(f"\nAn error occurred while saving the production model: {e}")




--- Step 2: Loading and Combining Data ---
  - Total Rows: 100000
  - Total Columns: 16

First 5 rows of the combined data:
                          timestamp     agent_name  agent_ip srcuser  srcip  \
0  2025-10-25T04:39:03.181234+00:00      dev-vm-01       NaN   irene    NaN   
1  2025-10-25T04:39:07.181234+00:00  prod-server-2       NaN    nina    NaN   
2  2025-10-25T04:39:11.181234+00:00      dev-vm-01       NaN    tina    NaN   
3  2025-10-25T04:39:17.181234+00:00    sales-host2       NaN     bob    NaN   
4  2025-10-25T04:39:22.181234+00:00      dev-vm-01       NaN     bob    NaN   

  decoder_name program_name  rule_level                  rule_description  \
0         sshd         sshd           3  sshd: Accepted password for user   
1         sshd         sshd           3  sshd: Accepted password for user   
2          pam         sudo           3          PAM: User login success.   
3          pam         sudo           3          PAM: User login success.   
4         sshd 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Performance Metrics (Checking for Overfitting):
  - Training Accuracy: 0.9902  |  Testing Accuracy: 0.9890
  - Training Loss:     0.0312  |  Testing Loss:     0.0401

--------------------
Full Test Set Evaluation:
Classification Report:
               precision    recall  f1-score   support

   benign (0)       1.00      0.97      0.99     12000
malicious (1)       0.98      1.00      0.99     18000

     accuracy                           0.99     30000
    macro avg       0.99      0.99      0.99     30000
 weighted avg       0.99      0.99      0.99     30000

Confusion Matrix:
[[11697   303]
 [   27 17973]]

--- Example of Final Output Format (Head of Test Set) ---
       label  score (prob_malicious)
0  malicious                0.975339
1  malicious                0.988315
2  malicious                0.991831
3  malicious                0.983905
4  malicious                0.992441


--- Step 7: Interpretation of Overfitting/Underfitting ---
--- Model Loss Comparison ---
  Logisti